In [1]:
import wandb
import pandas as pd
import os
from tqdm import tqdm

# from table_plotter import print_result_table

In [2]:
api = wandb.Api(timeout=600)


In [3]:
# Specify cache directory
cache_dir = "./wandb_cache"
os.makedirs(cache_dir, exist_ok=True)

In [4]:

skipped_runs = []  # List to store IDs of skipped runs

evaluation_keys = ['Evaluation/acc_imp_perc', 'Evaluation/exist_imp_perc', 'Evaluation/reach_imp_perc', 'Evaluation/path_length',
                   'Evaluation/fn_imp_perc', 'Evaluation/fp_imp_perc', 'Evaluation/tn_imp_perc', 'Evaluation/tp_imp_perc', 
                   'Evaluation/solvability', 'Evaluation/playability']
evaluation2_keys = ['Evaluation/playability', 'Evaluation/naive_playability', 'Evaluation/solvability', 'Evaluation/acc_imp_perc']


In [5]:
def get_dataframe_from_run(run):
    dfs = list()
    
    for run in tqdm(runs):
    
        # Define cache filename based on run ID
        cache_file = os.path.join(cache_dir, f"{run.id}.csv")
        
        # Check if cached file exists
        if os.path.exists(cache_file):
            # Load cached DataFrame
            df = pd.read_csv(cache_file)
        else:
            if run.state == "running":
                print(f"Skipping run ID: {run.id} (state: {run.state})")
                continue
            
            df = run.history(keys=["Evaluation/llm_iteration", *evaluation_keys[:1]])
    
            def append_key(src_df, key):
    
                tgt_df = run.history(keys=[key, "Evaluation/llm_iteration"])
                src_df = pd.merge(src_df, tgt_df, on="Evaluation/llm_iteration", how="outer")
                src_df = src_df.drop(columns=["_step_x", "_step_y"], errors="ignore")
                return src_df
    
            for key in evaluation_keys[1:]:
                try:
                    df = append_key(df, key)
                except Exception as e:
                    print(f"Error: {e} at run ID: {run.id}")
    
            
            # Add run config to DataFrame with prefix 'config.'
            for key, value in run.config.items():
                if isinstance(value, list):
                    value = ",".join(map(str, value))  # Convert list to comma-separated string
                df[key] = value
    
            # 기본값 설정
            default_values = {'n_self_alignment': 0, 'feedback_type': 'default'}
            # 열이 없을 경우 기본값으로 채워 넣기
            for col, value in default_values.items():
                if col not in df.columns:
                    df[col] = value
            
             
            # Filter columns
            key_filter = ['run_id', 'final_state', 'target_character', 'pe', 'gpt_model', 'branch_factor', 'exp_name', 'evaluator', 'total_iterations', 'n_self_alignment', 'feedback_type', 'feedback_input_type', 'total_timesteps', 
                          'reward_feature', 'fewshot', 'problem', 'seed', 
                          'Evaluation/llm_iteration'] + evaluation_keys
            auxiliary_key_filter = []
            
            df['run_id'] = run.id  # Add run ID as a column
            df['final_state'] = run.state
            
            try:
                df = df[key_filter + auxiliary_key_filter]
            except KeyError:
                df = df[key_filter]
            
            # Save DataFrame to cache as CSV
            df.to_csv(cache_file, index=False)
        
        dfs.append(df)
    
    # Concatenate all DataFrames
    df = pd.concat(dfs, ignore_index=True)
    
    return df

In [6]:
def get_dataframe_from_run2(run):
    dfs = []
    
    for run in tqdm(runs):
    
        # Define cache filename based on run ID
        cache_file = os.path.join(cache_dir, f"{run.id}.csv")
        
        # Check if cached file exists
        if os.path.exists(cache_file):
            # Load cached DataFrame
            df = pd.read_csv(cache_file)
        else:
            if run.state == "running":
                print(f"Skipping run ID: {run.id} (state: {run.state})")
                continue
            
            df = run.history(keys=["Evaluation/llm_iteration", *evaluation2_keys[:1]])
    
            def append_key(src_df, key):
    
                tgt_df = run.history(keys=[key, "Evaluation/llm_iteration"])
                src_df = pd.merge(src_df, tgt_df, on="Evaluation/llm_iteration", how="outer")
                src_df = src_df.drop(columns=["_step_x", "_step_y"], errors="ignore")
                return src_df
    
            for key in evaluation2_keys[1:]:
                try:
                    df = append_key(df, key)
                except Exception as e:
                    print(f"Error: {e} at run ID: {run.id}")
    
            
            # Add run config to DataFrame with prefix 'config.'
            for key, value in run.config.items():
                if isinstance(value, list):
                    value = ",".join(map(str, value))  # Convert list to comma-separated string
                df[key] = value
    
            # 기본값 설정
            default_values = {'n_self_alignment': 0, 'feedback_type': 'default'}
            # 열이 없을 경우 기본값으로 채워 넣기
            for col, value in default_values.items():
                if col not in df.columns:
                    df[col] = value
            
             
            # Filter columns
            key_filter = ['run_id', 'final_state', 'target_character', 'pe', 'gpt_model', 'branch_factor', 'exp_name', 'evaluator', 'total_iterations', 'n_self_alignment', 'feedback_type', 'feedback_input_type', 'total_timesteps', 
                          'reward_feature', 'fewshot', 'problem', 'seed', 
                          'Evaluation/llm_iteration'] + evaluation2_keys
            auxiliary_key_filter = []
            
            df['run_id'] = run.id  # Add run ID as a column
            df['final_state'] = run.state
            
            try:
                df = df[key_filter + auxiliary_key_filter]
            except KeyError:
                df = df[key_filter]
            
            # Save DataFrame to cache as CSV
            df.to_csv(cache_file, index=False)
        
        dfs.append(df)
    
    # Concatenate all DataFrames
    df = pd.concat(dfs, ignore_index=True)
    
    return df

In [7]:
runs = api.runs("inchangbaek4907/scenario-llmeval")
scenario_df = get_dataframe_from_run(runs)
scenario_df = scenario_df[scenario_df['Evaluation/llm_iteration'] <= 6]
# set score column with acc_imp_perc
scenario_df['score'] = scenario_df['Evaluation/acc_imp_perc']
scenario_df

100%|██████████| 20/20 [01:36<00:00,  4.84s/it]


,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/exist_imp_perc,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score
0,pqy95q71,finished,2,got,gpt-4o,2,llmeval,llm,6,0,...,0.633333,0.550000,27.000000,2.266667,0.000000,0.433333,0.300000,0.433333,0.800000,0.244444
1,pqy95q71,finished,2,got,gpt-4o,2,llmeval,llm,6,0,...,0.866667,0.183333,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,pqy95q71,finished,2,got,gpt-4o,2,llmeval,llm,6,0,...,0.983333,0.283333,26.000000,2.933333,0.033333,0.000000,0.033333,0.000000,0.033333,0.011111
3,pqy95q71,finished,2,got,gpt-4o,2,llmeval,llm,6,0,...,0.983333,0.283333,26.000000,2.933333,0.033333,0.000000,0.033333,0.000000,0.033333,0.011111
4,pqy95q71,finished,2,got,gpt-4o,2,llmeval,llm,6,0,...,0.566667,0.016667,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,1f9wvxa9,finished,1,got,gpt-4o,2,llmeval,llm,6,0,...,0.466667,0.166667,29.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.066667,0.000000
116,1f9wvxa9,finished,1,got,gpt-4o,2,llmeval,llm,6,0,...,0.466667,0.166667,29.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.066667,0.000000
117,1f9wvxa9,finished,1,got,gpt-4o,2,llmeval,llm,6,0,...,0.666667,0.566667,26.526316,2.766667,0.200000,0.000000,0.033333,0.100000,0.633333,0.011111
118,1f9wvxa9,finished,1,got,gpt-4o,2,llmeval,llm,6,0,...,0.133333,0.133333,26.000002,2.866667,0.133333,0.000000,0.000000,0.066667,1.000000,0.000000


In [8]:
runs = api.runs("inchangbaek4907/scenario2-llmeval")
scenario2_df = get_dataframe_from_run2(runs)
scenario2_df = scenario2_df[scenario2_df['Evaluation/llm_iteration'] <= 6]
# set score column with playability
scenario2_df['score'] = scenario2_df['Evaluation/playability']
scenario2_df

100%|██████████| 39/39 [01:16<00:00,  1.96s/it]


,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,reward_feature,fewshot,problem,seed,Evaluation/llm_iteration,Evaluation/playability,Evaluation/naive_playability,Evaluation/solvability,Evaluation/acc_imp_perc,score
0,85qw28n7,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,array,False,dungeon4,9,1,0.866667,0.900000,0.900000,0.900000,0.866667
1,85qw28n7,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,array,False,dungeon4,9,2,0.866667,0.900000,0.900000,0.900000,0.866667
2,85qw28n7,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,array,False,dungeon4,9,3,0.000000,0.000000,0.200000,0.200000,0.000000
3,85qw28n7,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,array,False,dungeon4,9,4,0.000000,0.000000,0.200000,0.200000,0.000000
4,85qw28n7,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,array,False,dungeon4,9,5,0.533333,0.600000,0.600000,0.633333,0.533333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229,9t69ed5l,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,array,False,dungeon4,2,2,0.000000,0.100000,0.133333,0.233333,0.000000
230,9t69ed5l,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,array,False,dungeon4,2,3,0.000000,0.100000,0.100000,0.300000,0.000000
231,9t69ed5l,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,array,False,dungeon4,2,4,0.000000,0.100000,0.100000,0.300000,0.000000
232,9t69ed5l,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,array,False,dungeon4,2,5,0.000000,0.000000,0.033333,0.133333,0.000000


In [9]:
scenario_df = pd.concat([scenario_df, scenario2_df], ignore_index=True)
scenario_df

,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score,Evaluation/naive_playability
0,pqy95q71,finished,2,got,gpt-4o,2,llmeval,llm,6,0,...,0.550000,27.0,2.266667,0.000000,0.433333,0.300000,0.433333,0.800000,0.244444,NaN
1,pqy95q71,finished,2,got,gpt-4o,2,llmeval,llm,6,0,...,0.183333,0.0,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
2,pqy95q71,finished,2,got,gpt-4o,2,llmeval,llm,6,0,...,0.283333,26.0,2.933333,0.033333,0.000000,0.033333,0.000000,0.033333,0.011111,NaN
3,pqy95q71,finished,2,got,gpt-4o,2,llmeval,llm,6,0,...,0.283333,26.0,2.933333,0.033333,0.000000,0.033333,0.000000,0.033333,0.011111,NaN
4,pqy95q71,finished,2,got,gpt-4o,2,llmeval,llm,6,0,...,0.016667,0.0,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349,9t69ed5l,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.133333,0.000000,0.000000,0.100000
350,9t69ed5l,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.100000,0.000000,0.000000,0.100000
351,9t69ed5l,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.100000,0.000000,0.000000,0.100000
352,9t69ed5l,finished,5,tot,gpt-4o,2,llmeval,llm,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.033333,0.000000,0.000000,0.000000


In [10]:
# Print summary of skipped runs
print("\nSummary of Skipped Runs:")
print(f"Total skipped runs: {len(skipped_runs)}")
print("Skipped run IDs:", skipped_runs)


Summary of Skipped Runs:
Total skipped runs: 0
Skipped run IDs: []


In [11]:
df = pd.concat([scenario_df], ignore_index=True)

In [12]:
df.to_csv(f"llmeval_result.csv", index=False)